# Multi-cohort EWAS with surrogate variable tuning

This notebook adds a paper-inspired SVA use case modeled after a multi-cohort Alzheimer's disease methylation EWAS. It does **not** reproduce the original study data. Instead, it mirrors several methodological ideas in a synthetic setting:

- brain-region style tissue labels
- Braak-stage-like continuous outcome
- age and sex adjustment
- cohort-level heterogeneity
- hidden technical and biological variation
- surrogate variable estimation and tuning using inflation control


In [ ]:
options(stringsAsFactors = FALSE)
set.seed(20260413)

check_package <- function(pkg) requireNamespace(pkg, quietly = TRUE)
have_sva <- check_package("sva")

pca_surrogates <- function(expr, n_sv = 2) {
  expr_centered <- t(scale(t(as.matrix(expr)), center = TRUE, scale = FALSE))
  pc <- prcomp(t(expr_centered), center = FALSE, scale. = FALSE)
  sv <- pc$x[, seq_len(min(n_sv, ncol(pc$x))), drop = FALSE]
  colnames(sv) <- paste0("PC_SV_", seq_len(ncol(sv)))
  sv
}

estimate_surrogates <- function(expr, mod, mod0, n_sv = 3) {
  if (have_sva) {
    fit <- tryCatch(sva::sva(as.matrix(expr), mod = mod, mod0 = mod0, n.sv = n_sv), error = function(e) NULL)
    if (!is.null(fit) && !is.null(fit$sv)) {
      sv <- fit$sv
      colnames(sv) <- paste0("SV", seq_len(ncol(sv)))
      return(list(method = "sva", sv = sv))
    }
  }
  list(method = "pca_fallback", sv = pca_surrogates(expr, n_sv = n_sv))
}

fit_featurewise_lm <- function(expr, design, coef_index = 2) {
  expr <- as.matrix(expr)
  XtX_inv <- solve(crossprod(design))
  betas <- XtX_inv %*% t(design) %*% t(expr)
  fitted <- design %*% betas
  resid <- t(expr) - fitted
  df_resid <- nrow(design) - ncol(design)
  sigma2 <- colSums(resid^2) / df_resid
  se_beta <- sqrt(sigma2 * XtX_inv[coef_index, coef_index])
  t_stat <- betas[coef_index, ] / se_beta
  p_val <- 2 * pt(abs(t_stat), df = df_resid, lower.tail = FALSE)
  out <- data.frame(beta = as.numeric(betas[coef_index, ]), p = as.numeric(p_val))
  out$padj <- p.adjust(out$p, method = "BH")
  out
}

evaluate_hits <- function(res, truth) {
  detected <- res$padj < 0.05
  tp <- sum(detected & truth)
  fp <- sum(detected & !truth)
  fn <- sum(!detected & truth)
  tn <- sum(!detected & !truth)
  data.frame(
    TP = tp, FP = fp, FN = fn, TN = tn,
    sensitivity = tp / (tp + fn),
    precision = ifelse(tp + fp == 0, NA, tp / (tp + fp)),
    empirical_FDR = ifelse(tp + fp == 0, NA, fp / (tp + fp))
  )
}

estimate_lambda <- function(pvals) {
  chisq <- qchisq(1 - pvals, df = 1)
  median(chisq, na.rm = TRUE) / qchisq(0.5, 1)
}


## Simulation design

We simulate a meta-analysis-like EWAS where each sample belongs to a cohort and a brain region. The primary phenotype is a continuous **Braak-stage-like burden score**. Hidden variation is introduced to resemble cell-composition shifts, chip or plate effects, and cohort-level technical differences.


In [ ]:
# Cohort and tissue structure
n_cpg <- 15000
cohorts <- c("London_like", "ROSMAP_like", "MountSinai_like")
tissues <- c("PFC", "Temporal", "Entorhinal", "Cerebellum")
samples_per_cohort <- c(64, 58, 62)

cohort <- factor(rep(cohorts, samples_per_cohort))
n_samples <- length(cohort)
tissue <- factor(sample(tissues, n_samples, replace = TRUE, prob = c(0.34, 0.27, 0.17, 0.22)))
age <- round(rnorm(n_samples, mean = 79, sd = 7))
sex <- factor(sample(c("F", "M"), n_samples, replace = TRUE))

# Continuous Braak-style pathology score
braak_stage <- scale(
  rnorm(n_samples) +
    ifelse(tissue == "Entorhinal", 0.9, 0) +
    ifelse(tissue == "Temporal", 0.35, 0) +
    ifelse(cohort == "MountSinai_like", 0.20, 0)
)[, 1]

# Hidden nuisance factors
hidden_plate <- scale(rnorm(n_samples) + as.numeric(cohort) * 0.8 + ifelse(tissue == "Cerebellum", -0.5, 0))[, 1]
hidden_cellmix <- scale(rnorm(n_samples) + braak_stage * 0.45 + ifelse(tissue == "PFC", 0.25, 0))[, 1]
hidden_bisulfite <- scale(rnorm(n_samples) + as.numeric(tissue) * 0.20)[, 1]

# Probe truth labels
truth <- rep(FALSE, n_cpg)
truth[sample(seq_len(n_cpg), 900)] <- TRUE

baseline <- rnorm(n_cpg, mean = 0, sd = 0.65)
braak_effect <- rep(0, n_cpg)
braak_effect[truth] <- rnorm(sum(truth), mean = 0.055, sd = 0.018)

cohort_loading <- rnorm(n_cpg, 0, 0.12)
cell_loading <- rnorm(n_cpg, 0, 0.10)
bisul_loading <- rnorm(n_cpg, 0, 0.08)
age_loading <- rnorm(n_cpg, 0, 0.002)
sex_loading <- rnorm(n_cpg, 0, 0.015)

meth_beta <- matrix(0, nrow = n_cpg, ncol = n_samples)
for (i in seq_len(n_cpg)) {
  eta <- baseline[i] +
    braak_effect[i] * braak_stage +
    cohort_loading[i] * hidden_plate +
    cell_loading[i] * hidden_cellmix +
    bisul_loading[i] * hidden_bisulfite +
    age_loading[i] * age +
    sex_loading[i] * (sex == "M") +
    rnorm(n_samples, sd = 0.05)
  meth_beta[i, ] <- pmin(pmax(plogis(eta), 1e-4), 1 - 1e-4)
}

m_values <- qlogis(meth_beta)
rownames(m_values) <- paste0("cg", sprintf("%08d", seq_len(n_cpg)))
colnames(m_values) <- paste0("sample_", seq_len(n_samples))

meta <- data.frame(
  sample = colnames(m_values),
  cohort = cohort,
  tissue = tissue,
  age = age,
  sex = sex,
  braak_stage = braak_stage,
  hidden_plate = hidden_plate,
  hidden_cellmix = hidden_cellmix,
  hidden_bisulfite = hidden_bisulfite
)

cat("Samples:", n_samples, "
")
cat("CpGs:", n_cpg, "
")
cat("True Braak-associated CpGs:", sum(truth), "
")


## Tuning surrogate variables using inflation control

The paper-inspired part of this workflow is not just adding surrogate variables, but **tuning how many to include**. We compare a naive model, a measured-covariates model, and a sequence of SVA-adjusted models. The chosen model is the first one that brings lambda below 1.2, or otherwise the lowest-lambda model.


In [ ]:
mod_naive <- model.matrix(~ braak_stage, data = meta)
mod_measured <- model.matrix(~ braak_stage + age + sex + cohort + tissue, data = meta)
mod0_measured <- model.matrix(~ age + sex + cohort + tissue, data = meta)

res_naive <- fit_featurewise_lm(m_values, mod_naive, coef_index = 2)
res_measured <- fit_featurewise_lm(m_values, mod_measured, coef_index = 2)

lambda_naive <- estimate_lambda(res_naive$p)
lambda_measured <- estimate_lambda(res_measured$p)

cat("Naive lambda:", round(lambda_naive, 3), "
")
cat("Measured-covariates lambda:", round(lambda_measured, 3), "
")

tuning <- data.frame(n_sv = integer(), lambda = numeric(), method = character(), stringsAsFactors = FALSE)
best_lambda <- Inf
best_k <- NA
best_sv <- NULL
best_res <- NULL
best_method <- NA

for (k in 1:8) {
  svfit <- estimate_surrogates(m_values, mod_measured, mod0_measured, n_sv = k)
  design_k <- cbind(mod_measured, svfit$sv)
  res_k <- fit_featurewise_lm(m_values, design_k, coef_index = 2)
  lambda_k <- estimate_lambda(res_k$p)
  tuning <- rbind(tuning, data.frame(n_sv = k, lambda = lambda_k, method = svfit$method))
  if (lambda_k < best_lambda) {
    best_lambda <- lambda_k
    best_k <- k
    best_sv <- svfit$sv
    best_res <- res_k
    best_method <- svfit$method
  }
  if (lambda_k < 1.2) break
}

print(tuning)
cat("Selected SV count:", best_k, "
")
cat("Selected method:", best_method, "
")
cat("Selected lambda:", round(best_lambda, 3), "
")


In [ ]:
eval_naive <- evaluate_hits(res_naive, truth)
eval_measured <- evaluate_hits(res_measured, truth)
eval_sva <- evaluate_hits(best_res, truth)

print(rbind(
  cbind(model = "naive", eval_naive),
  cbind(model = "measured", eval_measured),
  cbind(model = "measured_plus_sv", eval_sva)
))

oldpar <- par(no.readonly = TRUE)
par(mfrow = c(2, 2))

hist(res_naive$p, breaks = 40, main = paste0("Naive p-values, lambda=", round(lambda_naive, 2)), xlab = "p")
hist(res_measured$p, breaks = 40, main = paste0("Measured covariates, lambda=", round(lambda_measured, 2)), xlab = "p")
plot(tuning$n_sv, tuning$lambda, type = "b", pch = 16,
     main = "Inflation versus number of surrogate variables",
     xlab = "Number of SVs", ylab = expression(lambda))
abline(h = 1.2, col = "red", lty = 2)

obs <- sort(best_res$p)
exp <- ppoints(length(obs))
plot(-log10(exp), -log10(obs), pch = 16, cex = 0.35,
     main = paste0("QQ plot for selected model (k=", best_k, ")"),
     xlab = "Expected -log10(p)", ylab = "Observed -log10(p)")
abline(0, 1, col = "red", lty = 2)
par(oldpar)


In [ ]:
# Relate estimated SVs back to known hidden factors
if (!is.null(best_sv)) {
  sv_cor <- sapply(seq_len(ncol(best_sv)), function(j) {
    c(
      cor(best_sv[, j], meta$hidden_plate),
      cor(best_sv[, j], meta$hidden_cellmix),
      cor(best_sv[, j], meta$hidden_bisulfite),
      cor(best_sv[, j], meta$braak_stage)
    )
  })
  sv_cor <- t(sv_cor)
  colnames(sv_cor) <- c("hidden_plate", "hidden_cellmix", "hidden_bisulfite", "braak_stage")
  rownames(sv_cor) <- paste0("SV", seq_len(nrow(sv_cor)))
  print(round(sv_cor, 3))
}

head(best_res[order(best_res$p), ], 12)


## Interpretation

This notebook mirrors several design ideas from the paper without attempting a direct replication. It is useful for showing why methylation EWAS can benefit from both measured-covariate adjustment and inflation-aware surrogate variable tuning, especially in multi-cohort brain tissue settings.
